In [ ]:
import os
import sys
import yaml
sys.path.append(os.path.abspath('../src'))

from utils import *
from keras.applications.resnet50 import preprocess_input as resnet_preprocess

# Load config (relative to notebooks/)
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
# 1. Carregar o histórico guardado (usando o mesmo caminho)
resnet_history_path = os.path.join('..', config['paths']['models_dir'], 'resnet50', 'resnet50_history.json')
resnet_history = load_history(resnet_history_path)

# 2. Plottar as curvas de treino
plot_learning_curves(resnet_history, title="Curvas de Treino: ResNet50 (Fase 1 + Fine-Tuning)")


In [ ]:
# 1. Carregar o histórico guardado (usando o mesmo caminho)
scratch_history_path = os.path.join('..', config['paths']['models_dir'], 'scratch', 'scratch_history.json')
scratch_history = load_history(scratch_history_path)

# 2. Plottar as curvas de treino
plot_learning_curves(scratch_history, title="Scratch Curvas de Treino")


In [ ]:
model_scratch_path = os.path.join('..', config['paths']['models_dir'], 'scratch', 'scratch_best.keras')
model_resnet_path = os.path.join('..', config['paths']['models_dir'], 'resnet50', 'resnet50_best.keras')
model_scratch = keras.models.load_model(model_scratch_path)
model_resnet = keras.models.load_model(model_resnet_path)

In [ ]:
# All paths relative to notebooks/
train_dir = os.path.join('..', config['paths']['train_dir'])
val_dir = os.path.join('..', config['paths']['val_dir'])
test_dir = os.path.join('..', config['paths']['test_dir'])
IMG_SIZE = tuple(config['img_size'])
BATCH_SIZE = config['batch_size']

_, _, test_ds, class_names = load_datasets(
    train_dir, val_dir, test_dir,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# Carregar modelos
model_scratch = keras.models.load_model("models_results/scratch/scratch_best.keras")
model_resnet  = keras.models.load_model("models_results/resnet50/resnet50_best.keras")

# Avaliar — mesmo test_ds para os dois
metrics_cnn    = evaluate_model(model_scratch, test_ds, class_names, model_name="CNN Scratch")
metrics_resnet = evaluate_model(model_resnet,  test_ds, class_names, model_name="ResNet50 Fine-tuned")

compare_models([metrics_cnn, metrics_resnet])